In [ ]:
# Can also download .josn
import json
from collections import Counter

from datasets import load_dataset

In [ ]:
ds = load_dataset("Salesforce/xlam-function-calling-60k")

In [ ]:
keys = ds["train"][0].keys()
print(len(ds["train"]))
keys

In [ ]:
ds["train"][0]

Inspect types for each call. Builds input specs for Converter

In [ ]:
for key in keys:
    print(f"{key}: {type(ds["train"][0][key])}")

In [ ]:
tool_types = Counter()
tool_params = Counter()
for i in range(60_000):
    tools = json.loads(ds["train"][i]["tools"])
    for tool in tools:
        # print("\n=== TOOL === ")
        # print("Name:", tool['name'])
        # print("Description:", tool["description"])
        # print("Parameters:")
        for parameter, t in tool["parameters"].items():
            for k in t.keys():
                tool_params[k] += 1
            # print("    name:", parameter)
            # print("    type:", t["type"])
            tool_types[t["type"]] += 1
print("Tool types: ", tool_types)
print("Tool keys:", tool_params)


In [ ]:
json.loads(ds["train"][0]["tools"])[0]["parameters"]

XLAM documentation claims a `required` key exists in tool parameters, but none is found. Will have to parse for optional inputs.

## Test Normalization

In [ ]:
from collections import Counter
from tool_forge.normalize import normalize_row
from tool_forge.verify import verify, MalformedSpecError

outcomes: Counter[str] = Counter()
normalize_errors: Counter[str] = Counter()
total_calls = 0

for row in ds["train"]:
    try:
        conv = normalize_row(row)
    except Exception as e:                      # survey-only broad catch (categorize, don't stop)
        normalize_errors[type(e).__name__] += 1
        continue
    for call in conv.gold_calls:
        total_calls += 1
        try:
            outcomes[verify(call, conv.tools).result.value] += 1
        except MalformedSpecError:
            outcomes["MALFORMED_SPEC"] += 1

print("normalize errors:", normalize_errors)
print("verify outcomes:", outcomes)
print(f"quarantine rate: {1 - outcomes['VALID'] / total_calls:.2%}  ({total_calls} gold calls)")
